# Subject-specific design matrices (amblyo7T)

The pRF fit (`prf_cssfit_gpu`) loads `derivatives/vdm/sub-XX/sub-XX_task-{task}_vdm.npy` when it exists, otherwise the common design `derivatives/vdm/task-{task}_vdm.npy`. This notebook builds a design for **every subject** (sub-07 excluded) and for the 6 analysed tasks.

**Bars sequence of each run** (audit):

| subject | Bars LE run-01 | Bars LE run-02 | Bars RE run-01 | Bars RE run-02 |
|---|---|---|---|---|
| sub-14, sub-17 | B | B | B | B |
| sub-05 | A | A | **B** | A |
| sub-16 | reconstructed | A | reconstructed | reconstructed |
| others | A | A | A | A |

**How**:
- Same conversion as `create_prf_videos.ipynb` + `create_vdm_files.ipynb` (.mat → 102×102 → one frame per TR → grey > 5 → 50×50 nearest).
- Concatenation order of `averaging.py`: Bars run-01, Bars run-02, Rings, Wedges.
- Sources:
  - A = `derivatives/vdm/original/Bars.mat`;
  - B = `sourcedata/C/jp/cond-1-stim-3.mat`;
  - sub-16 = corrected chains (`sub16_repeats_and_od.ipynb`), rebuilt from the sweeps of `Bars.mat`.
  - The rings / wedges blocks are copied from the current common design.

**Checks before writing anything**:
1. The conversion of `Bars.mat` must reproduce the current common design exactly.
2. Content of the common designs (which rings / wedges version).
3. sub-16 designs rebuilt identically.
4. Shapes, binary values, blocks, order; subjects all-A identical to the common design.
5. Data: for each bars run, the retained design must fit the data better than A and B (RingsWedges pRFs of `prfmodel_old`, independent of the bars).

**Writing**: `WRITE = False` (default) writes everything in `Desktop/fig_od/vdm_check/`. With `WRITE = True`, the designs are written in `derivatives/vdm/sub-XX/`, only if every check passed. Existing files are not overwritten (`OVERWRITE = False`).

In [ ]:
import os, glob, math, json
import numpy as np, pandas as pd, nibabel as nb
import scipy.io as sio
import matplotlib.pyplot as plt

WRITE = False        # False: everything is written in Desktop/fig_od/vdm_check. True: designs written in derivatives/vdm/sub-XX/

main = '/Users/uriel/disks/meso_shared/amblyo7T_prf'
vdm_dir = f'{main}/derivatives/vdm'
pp = f'{main}/derivatives/pp_data'
prf_ref_folder = 'prfmodel_old'          # RingsWedges pRFs used for the data check (independent of the bars)
check_dir = '/Users/uriel/Desktop/fig_od/vdm_check'
out_root = vdm_dir if WRITE else check_dir
os.makedirs(check_dir, exist_ok=True)

subjects = [f'sub-{i:02d}' for i in range(1, 22) if i != 7]          # sub-07 excluded (copy of sub-06)
tasks = ['BarsBarsRingsWedgesRightEye', 'BarsBarsRingsWedgesLeftEye', 'BarsBarsRightEye', 'BarsBarsLeftEye',
         'RingsWedgesRightEye', 'RingsWedgesLeftEye']
TR = 2.0

# Sequence of each bars run (audit: sub-14 / sub-17 = B; sub-05 RightEye run-01 = B; sub-16 = reconstructed except LeftEye run-02)
bars_runs = ['BarsLeftEye_run-01', 'BarsLeftEye_run-02', 'BarsRightEye_run-01', 'BarsRightEye_run-02']
sequence_of = {s: {r: 'A' for r in bars_runs} for s in subjects}
for s in ['sub-14', 'sub-17']:
    sequence_of[s] = {r: 'B' for r in bars_runs}
sequence_of['sub-05']['BarsRightEye_run-01'] = 'B'
for r in ['BarsLeftEye_run-01', 'BarsRightEye_run-01', 'BarsRightEye_run-02']:
    sequence_of['sub-16'][r] = 'sub16:' + r
print(pd.DataFrame(sequence_of).T.loc[['sub-05', 'sub-13', 'sub-14', 'sub-16', 'sub-17']].to_string())

# stimulus sources
A_mat = f'{vdm_dir}/original/Bars.mat'
B_mat = f'{main}/sourcedata/C/jp/cond-1-stim-3.mat'
sub16_final = '/Users/uriel/Desktop/fig_od/sub16_final'               # corrected chains (sub16_repeats_and_od.ipynb)


## Conversion (reproduces the current common design)

In [ ]:
# Conversion .mat -> 50 x 50 design, same steps as create_prf_videos.ipynb + create_vdm_files.ipynb:
# s.im (102 x 77 x 120) -> uint8(x 255) -> transpose -> 12 black rows on top, 13 at the bottom (102 x 102) -> one frame per TR
# -> grey > 5 -> resize to 50 x 50 with nearest neighbour. Two paths: direct (no video), or through an mp4 written and re-read
# exactly as the original notebooks (in case the video codec changes some pixels). The path that reproduces the current
# common design exactly is used.
try:
    import cv2
except ImportError:
    cv2 = None
print('opencv available:', cv2 is not None)


def load_im(fn):
    m = sio.loadmat(fn)
    return np.asarray(m['s']['im'][0, 0])


def pad_frame(frame):
    f = (frame * 255).astype(np.uint8).T                           # (77, 102)
    return np.vstack([np.zeros((12, 102), np.uint8), f, np.zeros((13, 102), np.uint8)])


def resize_nn(binary):
    if cv2 is not None:
        return cv2.resize(binary, dsize=(50, 50), interpolation=cv2.INTER_NEAREST)
    ii = np.floor(np.arange(50) * 102 / 50).astype(int)
    return binary[np.ix_(ii, ii)]


def convert_direct(im):
    out = np.zeros((50, 50, im.shape[-1]))
    for i in range(im.shape[-1]):
        out[..., i] = resize_nn((pad_frame(im[..., i]) > 5).astype(np.uint8))
    return out


def convert_video(im, tmp_fn):
    """Same as the original notebooks: mp4 at 60 Hz, 120 video frames per image, then the first frame of each TR."""
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    vw = cv2.VideoWriter(tmp_fn, fourcc, 60, (102, 102), False)
    for i in range(im.shape[-1]):
        f = pad_frame(im[..., i])
        for _ in range(120):
            vw.write(f)
    vw.release()
    cap = cv2.VideoCapture(tmp_fn)
    out, k, idx = np.zeros((50, 50, im.shape[-1])), 0, 0
    while cap.isOpened():
        ret, fr = cap.read()
        if not ret: break
        if idx % 120 == 0 and k < im.shape[-1]:
            g = cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY)
            out[..., k] = cv2.resize((g > 5).astype(np.uint8), dsize=(50, 50), interpolation=cv2.INTER_NEAREST)
            k += 1
        idx += 1
    cap.release()
    os.remove(tmp_fn)
    return out


common = {t: np.load(f'{vdm_dir}/task-{t}_vdm.npy') for t in tasks}
for t in tasks:
    print(f'common {t}: {common[t].shape}, values {np.unique(common[t])}')

A_im, B_im = load_im(A_mat), load_im(B_mat)
ref_A = common['BarsBarsRingsWedgesLeftEye'][..., :120]
convert = None
for name, fn in [('direct', convert_direct), ('video', (lambda im: convert_video(im, f'{check_dir}/tmp.mp4')) if cv2 else None)]:
    if fn is None: continue
    A_v = fn(A_im)
    n_diff = int((A_v != ref_A).sum())
    print(f'{name} conversion of Bars.mat vs current common design (A block): {n_diff} different pixels out of {A_v.size}')
    if n_diff == 0:
        convert = fn; break
assert convert is not None, 'STOP: no conversion path reproduces the current common design exactly.'
A_vdm = convert(A_im)
B_vdm = convert(B_im)
print('A vs B frame correlation: {:.3f}'.format(np.corrcoef(A_vdm.ravel(), B_vdm.ravel())[0, 1]))


## Content of the current common designs

In [ ]:
# What the current common designs contain, and which rings / wedges version they use
R_im, W_im = load_im(f'{vdm_dir}/original/Rings.mat'), load_im(f'{vdm_dir}/original/Wedges.mat')
R_vdm, W_vdm = convert(R_im), convert(W_im)
rows = []
for eye in ['LeftEye', 'RightEye']:
    full = common[f'BarsBarsRingsWedges{eye}']
    blocks = {'bars run-01': full[..., :120], 'bars run-02': full[..., 120:240], 'rings': full[..., 240:360], 'wedges': full[..., 360:480]}
    rows.append({'eye': eye,
                 'bars run-01 = A': np.array_equal(blocks['bars run-01'], A_vdm),
                 'bars run-02 = A': np.array_equal(blocks['bars run-02'], A_vdm),
                 'rings block = Rings.mat': np.array_equal(blocks['rings'], R_vdm),
                 'wedges block = Wedges.mat': np.array_equal(blocks['wedges'], W_vdm),
                 'BarsBars = first 240': np.array_equal(common[f'BarsBars{eye}'], full[..., :240]),
                 'RingsWedges = last 240': np.array_equal(common[f'RingsWedges{eye}'], full[..., 240:])})
    for folder in ['', 'real_rings-wedges/', 'bars_to_rings-wedges/']:
        for kind, blk in [('Rings', blocks['rings']), ('Wedges', blocks['wedges'])]:
            fn = f'{vdm_dir}/{folder}task-{kind}{eye}_vdm.npy'
            if os.path.exists(fn):
                v = np.load(fn)
                rows[-1][f'{folder or "top level/"}{kind} = block'] = (v.shape == blk.shape) and np.array_equal(v, blk)
print(pd.DataFrame(rows).set_index('eye').T.to_string())
print('\nThe rings / wedges blocks of the new designs are copied from the current common design (unchanged for everybody).')


## sub-16 reconstructed sequences

In [ ]:
# sub-16: the corrected chains (8 directions + a repeated 135-) rebuilt at the .mat level from the sweeps of Bars.mat (A contains
# the 8 directions), then converted with the same code. Check: identical to the designs of sub16_repeats_and_od.ipynb.
rr_, cc_ = np.meshgrid(np.arange(50) - 24.5, np.arange(50) - 24.5, indexing='ij')
X_pix, Y_pix = cc_, -rr_


def frame_label(f):
    m = f > 0
    if m.sum() < 3: return None, np.nan
    best = None
    for a in [0, 45, 90, 135]:
        th = np.deg2rad(a); p = X_pix[m] * np.cos(th) + Y_pix[m] * np.sin(th)
        if best is None or p.std() < best[0]: best = (p.std(), a, p.mean())
    return best[1], best[2]


def segment(stim):
    lab = [frame_label(stim[..., t]) for t in range(stim.shape[-1])]
    ev, t = [], 0
    while t < len(lab):
        o, p = lab[t]; u = t + 1
        if o is None:
            while u < len(lab) and lab[u][0] is None: u += 1
            ev.append(('b', t, u - t))
        else:
            while u < len(lab) and lab[u][0] == o and u - t < 12: u += 1
            ev.append((f"{o}{'+' if lab[u - 1][1] > p else '-'}", t, u - t))
        t = u
    return ev

sweep_start = {}
for k, s0, l in segment(A_vdm):
    if k != 'b' and l == 12 and k not in sweep_start:
        sweep_start[k] = s0
blank_idx = [i for i in range(120) if A_vdm[..., i].sum() == 0][0]
print('sweeps of A (start image):', sweep_start, '| blank image:', blank_idx)
assert len(sweep_start) == 8

sub16_vdm, sub16_chain = {}, {}
for r in ['BarsLeftEye_run-01', 'BarsRightEye_run-01', 'BarsRightEye_run-02']:
    target = np.load(f'{sub16_final}/sub-16_{r}_vdm.npy')
    chain = segment(target)
    im = np.repeat(A_im[..., blank_idx:blank_idx + 1], 120, axis=2).copy()
    for k, t0, l in chain:
        if k == 'b': continue
        im[..., t0:t0 + l] = A_im[..., sweep_start[k]:sweep_start[k] + l]
    v = convert(im)
    sub16_vdm[r], sub16_chain[r] = v, chain
    print(f"{r}: {' '.join(f'b{l}' if k == 'b' else k for k, t0, l in chain)} | identical to sub16_final: {np.array_equal(v, target)}")
    assert np.array_equal(v, target), 'STOP: the rebuilt sub-16 design differs from sub16_final'


def bars_design(s, r):
    seq = sequence_of[s][r]
    if seq == 'A': return A_vdm
    if seq == 'B': return B_vdm
    return sub16_vdm[seq.split(':')[1]]


## Designs of every subject + structural checks

In [ ]:
# Build the 6 designs of every subject + structural checks
def run_exists(s, r):
    return os.path.exists(f'{pp}/{s}/fsnative/func/fmriprep_dct_psc/{s}_ses-01_task-{r}_hemi-L_space-fsnative_fmriprep_dct_psc_bold.func.gii')

designs, struct_rows = {}, []
for s in subjects:
    d = {}
    for eye in ['LeftEye', 'RightEye']:
        b1, b2 = bars_design(s, f'Bars{eye}_run-01'), bars_design(s, f'Bars{eye}_run-02')
        rw = common[f'RingsWedges{eye}']
        d[f'BarsBarsRingsWedges{eye}'] = np.concatenate([b1, b2, rw], axis=2)
        d[f'BarsBars{eye}'] = np.concatenate([b1, b2], axis=2)
        d[f'RingsWedges{eye}'] = rw.copy()
    designs[s] = d
    all_A = all(sequence_of[s][r] == 'A' for r in bars_runs)
    rec = {'subject': s, 'sequences (LE1 LE2 RE1 RE2)': ' '.join(sequence_of[s][r].replace('sub16:', 'rec') for r in bars_runs),
           'all bars runs found': all(run_exists(s, r) for r in bars_runs),
           'rings/wedges runs found': all(run_exists(s, f'{k}{e}_run-01') for k in ['Rings', 'Wedges'] for e in ['LeftEye', 'RightEye']),
           'shapes ok': all(d[t].shape == common[t].shape for t in tasks),
           'binary': all(set(np.unique(d[t])) <= {0, 1} for t in tasks),
           'rings/wedges blocks = common': all(np.array_equal(d[f'BarsBarsRingsWedges{e}'][..., 240:], common[f'RingsWedges{e}'])
                                               for e in ['LeftEye', 'RightEye']),
           'BarsBars = first 240': all(np.array_equal(d[f'BarsBars{e}'], d[f'BarsBarsRingsWedges{e}'][..., :240]) for e in ['LeftEye', 'RightEye'])}
    rec['identical to common (expected only if all A)'] = all(np.array_equal(d[t], common[t]) for t in tasks)
    rec['check'] = rec['shapes ok'] and rec['binary'] and rec['rings/wedges blocks = common'] and rec['BarsBars = first 240'] and \
                   (rec['identical to common (expected only if all A)'] == all_A)
    struct_rows.append(rec)
struct = pd.DataFrame(struct_rows).set_index('subject')
print(struct.to_string())
print('\nAll structural checks passed:', bool(struct['check'].all()))
missing = struct[~struct['all bars runs found'] | ~struct['rings/wedges runs found']]
if len(missing):
    print('WARNING: some runs not found in fmriprep_dct_psc (the concatenation of these subjects may be shorter):', list(missing.index))


## Data check (each bars run)

In [ ]:
# Data check: for each bars run, the pRF prediction (RingsWedges pRFs of the same eye, prfmodel_old, independent of the bars)
# made with the retained design must fit the data better than with A and with B.
pix = 2 * math.degrees(math.atan(50.6 / 2 / 242)) / 50
Xd, Yd = X_pix * pix, Y_pix * pix
t_ = np.arange(0, 32, TR)
g = lambda a: t_ ** (a - 1) * np.exp(-t_) / math.gamma(a)
H = g(6) - g(16) / 6; H /= H.sum()
Hf = np.fft.rfft(H, n=256)


def conv_hrf(d):
    return np.fft.irfft(np.fft.rfft(d, n=256, axis=0) * Hf[:, None], n=256, axis=0)[:d.shape[0]]


def zc(a):
    a = a - a.mean(0)
    return a / (a.std(0) + 1e-9)


def fit_score(stim, y, P):
    x0, y0, sd, n = P
    G = np.exp(-((Xd.ravel()[:, None] - x0) ** 2 + (Yd.ravel()[:, None] - y0) ** 2) / (2 * sd ** 2))
    p = conv_hrf(np.clip(np.moveaxis(stim, -1, 0).reshape(120, -1) @ G, 0, None) ** n)
    best = -np.inf
    for sh in range(-2, 3):
        q = np.roll(p, sh, 0)
        if sh > 0: q[:sh] = p[0]
        elif sh < 0: q[sh:] = p[-1]
        best = max(best, float(np.median((zc(q) * zc(y)).mean(0))))
    return best

data_rows = []
for s in subjects:
    try:
        roi = np.concatenate([nb.load(glob.glob(f'{pp}/{s}/fsnative/rois/{s}_hemi-{h}_fmriprep_dct_*_rois-group-mmp.func.gii')[0]).darrays[0].data
                              for h in ['L', 'R']])
    except IndexError:
        print(s, ': no ROI file, skipped'); continue
    for eye in ['LeftEye', 'RightEye']:
        fns = [f'{pp}/{s}/fsnative/{prf_ref_folder}/prf_derivatives/{s}_task-RingsWedges{eye}_hemi-{h}_fmriprep_dct_psc_concat_prf-css_deriv.func.gii'
               for h in ['L', 'R']]
        if not all(os.path.exists(f) for f in fns):
            print(s, eye, ': no RingsWedges pRF, skipped'); continue
        D = np.hstack([np.vstack([a.data for a in nb.load(f).darrays]) for f in fns])
        for run in [1, 2]:
            r = f'Bars{eye}_run-0{run}'
            if not run_exists(s, r): continue
            hs = []
            for h in ['L', 'R']:
                dd = np.vstack([a.data for a in nb.load(
                    f'{pp}/{s}/fsnative/func/fmriprep_dct_psc/{s}_ses-01_task-{r}_hemi-{h}_space-fsnative_fmriprep_dct_psc_bold.func.gii').darrays])
                hs.append(dd if dd.shape[0] == 120 else dd.T)
            y = np.hstack(hs)
            idx = np.where((D[0] > 0.2) & (D[1] < 6) & np.isin(roi, [1, 2, 3]) & np.isfinite(y).all(0) & (y.std(0) > 0))[0]
            P = (D[7][idx], D[8][idx], np.clip(D[4][idx], 0.2, None), np.clip(D[11][idx], 0.05, 1))
            ret = bars_design(s, r)
            rec = {'subject': s, 'run': r, 'retained': sequence_of[s][r].replace('sub16:', 'reconstructed'),
                   'r retained': fit_score(ret, y[:, idx], P), 'r A': fit_score(A_vdm, y[:, idx], P), 'r B': fit_score(B_vdm, y[:, idx], P)}
            rec['retained >= best other - 0.01'] = rec['r retained'] >= max(rec['r A'], rec['r B']) - 0.01
            data_rows.append(rec)
    print(s, 'done', flush=True)
data_tab = pd.DataFrame(data_rows)
print(data_tab.round(3).to_string())
bad = data_tab[~data_tab['retained >= best other - 0.01']]
print('\nRuns where the retained design is not the best:', 'none' if bad.empty else '\n' + bad.round(3).to_string())
data_tab.to_csv(f'{check_dir}/vdm_data_check.tsv', sep='\t', index=False)


## Visual check

In [ ]:
show_s = ['sub-05', 'sub-13', 'sub-14', 'sub-16', 'sub-17']
frames = list(range(0, 120, 6))
fig, axes = plt.subplots(len(show_s) * 4, len(frames), figsize=(len(frames) * 0.6, len(show_s) * 4 * 0.65))
for i, s in enumerate(show_s):
    for j, r in enumerate(bars_runs):
        row = i * 4 + j
        v = bars_design(s, r)
        for c, t in enumerate(frames):
            axes[row, c].imshow(v[..., t], cmap='gray', vmin=0, vmax=1)
            axes[row, c].set_xticks([]); axes[row, c].set_yticks([])
            if row == 0: axes[row, c].set_title(f'{int(t * TR)} s', fontsize=6)
        axes[row, 0].set_ylabel(f"{s} {r.replace('Eye_run-0', ' ')}\n{sequence_of[s][r].replace('sub16:', 'rec')}", fontsize=5,
                                rotation=0, ha='right', va='center')
plt.tight_layout()
fig.savefig(f'{check_dir}/vdm_bars_runs.pdf', bbox_inches='tight')
plt.show()


## Save

In [ ]:
# Save. With WRITE = False: Desktop/fig_od/vdm_check/sub-XX/. With WRITE = True: derivatives/vdm/sub-XX/ (only if every check passed).
OVERWRITE = False
if WRITE:
    assert struct['check'].all(), 'STOP: structural checks failed, nothing written.'
    assert data_tab['retained >= best other - 0.01'].all(), 'STOP: data check failed for some runs, nothing written.'
n_written = 0
for s in subjects:
    d_out = f'{out_root}/{s}'
    os.makedirs(d_out, exist_ok=True)
    for t in tasks:
        fn = f'{d_out}/{s}_task-{t}_vdm.npy'
        if os.path.exists(fn) and WRITE and not OVERWRITE:
            print('exists, not overwritten:', fn); continue
        np.save(fn, designs[s][t]); n_written += 1
summary = pd.DataFrame({s: {r: sequence_of[s][r].replace('sub16:', 'reconstructed') for r in bars_runs} for s in subjects}).T
summary.to_csv(f'{check_dir}/bars_sequence_per_run.tsv', sep='\t')
print(f'{n_written} files written in {out_root} (WRITE = {WRITE})')
print('Sequence of each bars run saved in', f'{check_dir}/bars_sequence_per_run.tsv')
